In [ ]:

## Task 1: Data Collection

### Goal
Collect at least **100 open-licensed images** with their metadata.

### What You Need to Do

1. **Create a folder structure**:

   ```
   project/
   ├── images/           # Downloaded images
   ├── data/             # JSON metadata files
   └── project.ipynb     # Your notebook
   ```

2. **Find image sources** (choose one or more):
   - [Wikimedia Commons](https://commons.wikimedia.org/) - Use SPARQL queries (like in Practical 1)
   - [Unsplash API](https://unsplash.com/developers) - Free API for high-quality images
   - [Pexels API](https://www.pexels.com/api/) - Free stock photos
   - [Flickr API](https://www.flickr.com/services/api/) - Creative Commons images

3. **Download images programmatically** using the techniques from Practical 1, Exercise 6

4. **Extract and save metadata** for each image:
   - Image filename
   - Image dimensions (width, height)
   - File format (.jpg, .png, etc.)
   - File size (in KB)
   - Source URL
   - License information
   - EXIF data (if available): camera model, date taken, etc.

### Expected Output
- `images/` folder with 100+ images
- `data/images_metadata.json` containing metadata for all images

### Hints
- Use `PIL` to get image dimensions
- Use `os.path.getsize()` to get file size
- Use EXIF extraction (see Practical 2, Exercise 2)
- Store metadata as a list of dictionaries in JSON format

---

: 

In [6]:
import sys
import os
import json
import time
import shutil
import requests
import pandas as pd
from urllib.parse import unquote
from PIL import Image
from PIL.ExifTags import TAGS
from SPARQLWrapper import SPARQLWrapper, JSON as SPARQL_JSON

os.makedirs('images', exist_ok=True)
os.makedirs('data',   exist_ok=True)


ENDPOINT_URL = 'https://query.wikidata.org/sparql'

QUERIES = {
    'montagnes': '''
        SELECT DISTINCT ?mountain ?mountainLabel ?mountainRange ?mountainRangeLabel ?image WHERE {
          ?mountain wdt:P31 wd:Q8502 ;
                    wdt:P4552 ?mountainRange ;
                    wdt:P18 ?image .
          SERVICE wikibase:label { bd:serviceParam wikibase:language "en". }
        }
        LIMIT 25
    ''',
    'chats': '''
        SELECT DISTINCT ?itemLabel ?image WHERE {
          ?item wdt:P31 wd:Q146 ;
                wdt:P18 ?image .
          SERVICE wikibase:label { bd:serviceParam wikibase:language "en". }
        }
        LIMIT 25
    ''',
    'fleurs': '''
        SELECT DISTINCT ?itemLabel ?image WHERE {
          ?item wdt:P31 wd:Q506 ;
                wdt:P18 ?image .
          SERVICE wikibase:label { bd:serviceParam wikibase:language "en". }
        }
        LIMIT 25
    ''',
    'capitales': '''
        SELECT DISTINCT ?itemLabel ?image WHERE {
          ?item wdt:P31 wd:Q5119 ;
                wdt:P18 ?image .
          SERVICE wikibase:label { bd:serviceParam wikibase:language "en". }
        }
        LIMIT 25
    ''',
    'oiseaux': '''
        SELECT DISTINCT ?itemLabel ?image WHERE {
          ?item wdt:P31 wd:Q16521 ;
                wdt:P171* wd:Q5113 ;
                wdt:P18 ?image .
          SERVICE wikibase:label { bd:serviceParam wikibase:language "en". }
        }
        LIMIT 25
    '''
}


def get_results(endpoint_url, query):
    user_agent = 'WDQS-example Python/%s.%s' % (sys.version_info[0], sys.version_info[1])
    sparql = SPARQLWrapper(endpoint_url, agent=user_agent)
    sparql.setQuery(query)
    sparql.setReturnFormat(SPARQL_JSON)
    return sparql.query().convert()


all_images = []

for category, query in QUERIES.items():
    print(f'Requete SPARQL : {category}...')
    try:
        results = get_results(ENDPOINT_URL, query)
        bindings = results['results']['bindings']
        for result in bindings:
            if category == 'montagnes':
                label = result.get('mountainLabel', {}).get('value', '?')
                extra = result.get('mountainRangeLabel', {}).get('value', '')
            else:
                label = result.get('itemLabel', {}).get('value', '?')
                extra = ''
            all_images.append({
                'label':     label,
                'extra':     extra,
                'category':  category,
                'image_url': result['image']['value'],
            })
        print(f'  OK : {len(bindings)} images trouvees')
        time.sleep(2)
    except Exception as e:
        print(f'  Erreur : {e}')

print(f'\nTotal images collectees : {len(all_images)}')

def download_image(url, category, folder='images', max_retries=4):
    dest_folder = os.path.join(folder, category)
    os.makedirs(dest_folder, exist_ok=True)
    headers = {'User-Agent': 'Mozilla/5.0 (WikidataImageProject)'}

    for attempt in range(max_retries):
        try:
            request = requests.get(url, allow_redirects=True,
                                   headers=headers, stream=True, timeout=20)
            if request.status_code == 200:
                filename = unquote(os.path.basename(url.split('?')[0])).replace(' ', '_')
                filepath = os.path.join(dest_folder, filename)
                with open(filepath, 'wb') as image:
                    request.raw.decode_content = True
                    shutil.copyfileobj(request.raw, image)
                print('Saved to:', os.path.abspath(filepath))
                return filepath
            elif request.status_code == 429:
                retry_after = int(request.headers.get('Retry-After', 2 ** (attempt + 2)))
                print(f'  429 Rate limit - attente {retry_after}s (tentative {attempt+1}/{max_retries})')
                time.sleep(retry_after)
            else:
                print(f'Failed: {request.status_code} {url}')
                return None
        except Exception as e:
            wait = 2 ** (attempt + 1)
            print(f'  Erreur ({e}) - attente {wait}s')
            time.sleep(wait)

    print(f'  Echec definitif apres {max_retries} tentatives : {url}')
    return None


def extract_exif(pil_image):
    exif_data = {}
    try:
        raw_exif = pil_image._getexif()
        if raw_exif:
            for tag_id, value in raw_exif.items():
                tag = TAGS.get(tag_id, tag_id)
                if tag in ('Make', 'Model', 'DateTime', 'Software',
                           'Orientation', 'XResolution', 'YResolution'):
                    exif_data[tag] = str(value)
    except Exception:
        pass
    return exif_data


def extract_metadata(item):
    filepath = item['local_path']
    metadata = {
        'filename':     os.path.basename(filepath),
        'local_path':   filepath,
        'source_url':   item['image_url'],   # URL conservee ici
        'label':        item['label'],
        'extra':        item.get('extra', ''),
        'category':     item['category'],
        'license':      'Wikimedia Commons (voir page source)',
        'width':        None,
        'height':       None,
        'format':       None,
        'file_size_kb': None,
        'exif':         {}
    }
    try:
        metadata['file_size_kb'] = round(os.path.getsize(filepath) / 1024, 2)
        with Image.open(filepath) as img:
            metadata['width']  = img.size[0]
            metadata['height'] = img.size[1]
            metadata['format'] = img.format
            metadata['exif']   = extract_exif(img)
    except Exception as e:
        print(f'Impossible de lire {filepath} : {e}')
    return metadata


# Telechargement + extraction en meme temps
downloaded = []
all_metadata = []

for item in all_images:
    filepath = download_image(item['image_url'], item['category'])
    if filepath:
        item['local_path'] = filepath
        downloaded.append(item)
        all_metadata.append(extract_metadata(item))  # metadata extraite pendant que l'URL est dispo
    time.sleep(1)

print(f'{len(downloaded)} images telechargees.')
print(f'{len(all_metadata)} metadonnees extraites.')

# Sauvegarde JSON immediate
OUTPUT_JSON = os.path.join('data', 'images_metadata.json')
with open(OUTPUT_JSON, 'w', encoding='utf-8') as f:
    json.dump(all_metadata, f, ensure_ascii=False, indent=2)
print(f'Metadonnees sauvegardees dans : {OUTPUT_JSON}')

In [7]:
with open('data/images_metadata.json', 'r', encoding='utf-8') as f:
    all_metadata = json.load(f)

print(f'{len(all_metadata)} entrees chargees depuis le JSON.')

119 entrees chargees depuis le JSON.


In [8]:
df = pd.DataFrame(all_metadata)

# Aplatir la colonne 'exif' en colonnes separees
df = pd.concat(
    [df.drop(columns='exif'),
     pd.json_normalize(df['exif']).add_prefix('exif_')],
    axis=1
)

df.to_csv('data/images_metadata.csv', index=False)

print(f'DataFrame : {df.shape[0]} lignes x {df.shape[1]} colonnes')
df.head()

DataFrame : 119 lignes x 18 colonnes


,filename,local_path,source_url,label,extra,category,license,width,height,format,file_size_kb,exif_Make,exif_Model,exif_Software,exif_Orientation,exif_DateTime,exif_XResolution,exif_YResolution
0,Moosalp_Bonigersee.jpg,images/montagnes/Moosalp_Bonigersee.jpg,http://commons.wikimedia.org/wiki/Special:File...,Augstbordhorn,Pennine Alps,montagnes,Wikimedia Commons (voir page source),1280.0,887.0,JPEG,1563.03,Panasonic,DMC-TZ10,Adobe Photoshop Elements 3.0 Windows,1,2013:11:07 15:54:02,180.0,180.0
1,Hohtälli.JPG,images/montagnes/Hohtälli.JPG,http://commons.wikimedia.org/wiki/Special:File...,Hohtälli,Pennine Alps,montagnes,Wikimedia Commons (voir page source),1486.0,1031.0,JPEG,372.91,"OLYMPUS OPTICAL CO.,LTD","X-2,C-50Z",28-1011,1,2008:08:02 13:06:15,NaN,NaN
2,MontRogneux.jpeg,images/montagnes/MontRogneux.jpeg,http://commons.wikimedia.org/wiki/Special:File...,Mont Rogneux,Pennine Alps,montagnes,Wikimedia Commons (voir page source),4012.0,1276.0,JPEG,3525.20,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Roc_d'Orzival.jpg,images/montagnes/Roc_d'Orzival.jpg,http://commons.wikimedia.org/wiki/Special:File...,Roc d'Orzival,Pennine Alps,montagnes,Wikimedia Commons (voir page source),6240.0,4160.0,JPEG,15997.13,FUJIFILM,X-T30,Adobe Lightroom 5.5 (Windows),NaN,2022:09:15 20:18:22,240.0,240.0
4,Jagerhorn0001.jpg,images/montagnes/Jagerhorn0001.jpg,http://commons.wikimedia.org/wiki/Special:File...,Jägerhorn,Pennine Alps,montagnes,Wikimedia Commons (voir page source),600.0,443.0,JPEG,80.95,NaN,NaN,NaN,NaN,NaN,NaN,NaN
